# Using LangChain Python Library
Examples for non-streaming and streaming calls.


## Setup

In [ ]:
# imports
from dotenv import load_dotenv
load_dotenv(override=True)

# LangChain imports (third-party)
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama

# local helper
from streaming_utils import get_chunk_content

# depending on model, LangChain requires different instanciation
def get_llm_instance(model):
    if "gpt" in model or "o1" in model or "o3" in model:
        return ChatOpenAI(model=model)
    elif "gemini" in model:
        return ChatGoogleGenerativeAI(model=model)
    elif ":" in model:
        return ChatOllama(model=model)
    else:
        raise ValueError(f"Unknown model type for: {model}")


In [ ]:
# model
model_name = "llama3.2:1b"

# prepare message
system_message = "You are a helpful assistant."
user_message = "Tell me a joke about programming."

# LangChain uses its own message objects rather than simple dicts
messages = [SystemMessage(content=system_message), HumanMessage(content=user_message)]

## Non-Streaming

In [ ]:
def message(messages, model, **kwargs):
    llm_instance = get_llm_instance(model)

    used_kwargs = dict(kwargs)

    response = llm_instance.invoke(messages, **used_kwargs)

    return response.content

# simple test call
print(f"Using model: {model_name}")
response = message(messages, model_name)
print(response)


## Streaming

In [ ]:
def stream_message(messages, model_name, **kwargs):
    llm_instance = get_llm_instance(model_name)
    full_response = ""

    used_kwargs = dict(kwargs)

    for chunk in llm_instance.stream(messages, **used_kwargs):
        content = get_chunk_content(chunk)
        print(content, end="", flush=True)
        full_response += content
    return full_response


# simple test call
print(f"Using model: {model_name}")
response = stream_message(messages, model_name)